In [1]:
# Import packages and build dataframe
import io
import pandas as pd
import numpy as np

CSV = """site_id,date,water_temp_c,discharge_cfs,nitrate_mg_l,qc_flag
SB-01,2024-01-15,12.4,"1,240",0.82,good
SB-01,2024-02-15,13.1,"1,180",0.79,good
SB-01,2024-03-15,15.8,"2,050",1.21,good
SB-01,2024-04-15,18.2,"1,620",0.94,good
sb-01,2024-05-15,21.5,"980",0.66,good
SB-01 ,2024-06-15,-999,"540",0.51,suspect
SB-02,2024-01-15,11.8,"3,410",1.88,good
SB-02,2024-02-15,12.3,"3,290",1.92,good
SB-02,2024-03-15,14.9,"4,870",2.41,good
SB-02,2024-04-15,17.6,"3,950",2.09,estimated
SB-02,2024-05-15,20.9,"2,240",1.44,good
sb-02,2024-06-15,-999,"1,130",1.02,suspect
SB-03,2024-01-15,10.2,"620",<0.05,good
SB-03,2024-02-15,10.9,"590",<0.05,good
SB-03,2024-03-15,13.4,"1,040",0.33,good
SB-03,2024-04-15,16.1,"870",0.28,good
SB-03 ,2024-05-15,19.3,"410",<0.05,good
SB-03,2024-06-15,-999,"180",<0.05,suspect
"""

stream = pd.read_csv(io.StringIO(CSV))

In [2]:
print(stream)

   site_id        date  water_temp_c discharge_cfs nitrate_mg_l    qc_flag
0    SB-01  2024-01-15          12.4         1,240         0.82       good
1    SB-01  2024-02-15          13.1         1,180         0.79       good
2    SB-01  2024-03-15          15.8         2,050         1.21       good
3    SB-01  2024-04-15          18.2         1,620         0.94       good
4    sb-01  2024-05-15          21.5           980         0.66       good
5   SB-01   2024-06-15        -999.0           540         0.51    suspect
6    SB-02  2024-01-15          11.8         3,410         1.88       good
7    SB-02  2024-02-15          12.3         3,290         1.92       good
8    SB-02  2024-03-15          14.9         4,870         2.41       good
9    SB-02  2024-04-15          17.6         3,950         2.09  estimated
10   SB-02  2024-05-15          20.9         2,240         1.44       good
11   sb-02  2024-06-15        -999.0         1,130         1.02    suspect
12   SB-03  2024-01-15   

There are 3 monitoring sites in the data (SB-01, SB-02, SB-03)
The plausible range for streamwater temperature is between 0 and 30 degrees Celsius.
I expect a mean water temperature around 15 degrees Celsius

|What I expected| What happened| My hypothesis| What I changed| Result|
|:--------------|:-------------|:-------------|:--------------|:------|
|Discharge in m³/s, around 35|`TypeError: can't multiply sequence by non-int`|`discharge_cfs` is text because of thousands separators| `.str.replace(',','')` then `.astype(float)`| Works, 35.11 m³/s, guard added|
|Remove leading and trailing spaces from `site_id` |Spaces removed, but there are still two many unique site codes| Case is not consistent | `.str.strip()` then `.str.upper()` | Reduced to the three distinct sites|
|Convert `nitrate_mg_L` to floats| `ValueError: could not convert string to float: '<0.05'`| `nitrate_mg_L` value is not converitable to float because it is a logical (contains as < sign)| Convert values to 0.05 as the maximum potential value| `nitrate_mg_L` converted to float|
|Calculation of mean water temperature| `KeyError: 'temperature'`| `temperature` is not the correct name of the column| `stream['water_temp_c'].mean()` | Result is -153.8, an impossible value|
|Mean water temperature of about 15| Calculated -153.8, an impossible value| Missing values are encoded as -999| Changed entries where temperature is -999 to `NaN`| Calculated mean is 15.226|
| Access the last row of the dataframe| `IndexError: single positional indexer is out-of-bounds` | Index starts at 0, so the final entry has an index of 17 | `stream.iloc[17]` | Accessed the last row of the dataframe |


|Error Type |What it usually means in `pandas`|
|:-----------|:---------------------------------|
|KeyError| The name of the series/column you are trying to call is incorrect|
|AttributeError| The desired attribute is not part of the package you are using|
|TypeError| Data is of an unexpect type that doesn't work with the indicated operation|
|ValueError| A function cannot be applied to a particular value in a series, likely because of an indefinite value |
|IndexError| Indexing information given is not in the dataframe's index (out of range or wrong type of information)|

## 2. A worked de-bugging example

In [3]:
# Create the error
stream['discharge_cfs'] * 0.0283168

TypeError: can't multiply sequence by non-int of type 'float'

In [4]:
# Look at the data types
stream.dtypes

site_id           object
date              object
water_temp_c     float64
discharge_cfs     object
nitrate_mg_l      object
qc_flag           object
dtype: object

In [5]:
stream['discharge_cfs'].iloc[0] * 0.0283168   # same TypeError, one value
stream['discharge_cfs'].iloc[0]              # '1,240'

TypeError: can't multiply sequence by non-int of type 'float'

In [6]:
#Identify distinct values
stream['discharge_cfs'].unique()

array(['1,240', '1,180', '2,050', '1,620', '980', '540', '3,410', '3,290',
       '4,870', '3,950', '2,240', '1,130', '620', '590', '1,040', '870',
       '410', '180'], dtype=object)

In [7]:
# Fix and verify
discharge_cms = (stream['discharge_cfs']
                .str.replace(',', '', regex = False)
                .astype(float)) * 0.0283168
discharge_cms.head(3)

0    35.112832
1    33.413824
2    58.049440
Name: discharge_cfs, dtype: float64

In [8]:
# Add a guard
assert pd.api.types.is_numeric_dtype(discharge_cms), 'Discharge failed to convert to numeric'

In [9]:
# Minimum reproducible example:
import pandas as pd
df = pd.DataFrame({'discharge_cfs': ['1,240', '980']})
df['discharge_cfs'] * 0.0283168     # TypeError: can't multiply sequence by non-int

TypeError: can't multiply sequence by non-int of type 'float'

## 3 a

In [10]:
stream['site_id'].str.strip()

0     SB-01
1     SB-01
2     SB-01
3     SB-01
4     sb-01
5     SB-01
6     SB-02
7     SB-02
8     SB-02
9     SB-02
10    SB-02
11    sb-02
12    SB-03
13    SB-03
14    SB-03
15    SB-03
16    SB-03
17    SB-03
Name: site_id, dtype: object

In [11]:
stream['site_id'].unique()

array(['SB-01', 'sb-01', 'SB-01 ', 'SB-02', 'sb-02', 'SB-03', 'SB-03 '],
      dtype=object)

In [12]:
site_id_clean = (stream['site_id']
                 .str.strip()
                 .str.upper())

site_id_clean.unique()

array(['SB-01', 'SB-02', 'SB-03'], dtype=object)

## 3 b

In [13]:
stream['nitrate_mg_l'].astype(float)

ValueError: could not convert string to float: '<0.05'

In [14]:
# Adjust nitrate values reported as < 0.05 to the threshold value 0.05
stream.loc[(stream['nitrate_mg_l'] == '<0.05'), 'nitrate_mg_l'] = 0.05
stream['nitrate_mg_l'] = stream['nitrate_mg_l'].astype(float)
stream.dtypes

site_id           object
date              object
water_temp_c     float64
discharge_cfs     object
nitrate_mg_l     float64
qc_flag           object
dtype: object

## 3 c

In [15]:
stream['temperature'].mean()

KeyError: 'temperature'

In [16]:
stream['water_temp_c'].mean()

-153.8111111111111

In [17]:
stream['water_temp_c'].min()

-999.0

In [18]:
# Convert -999 to NaN (missing value designation)
stream.loc[(stream['water_temp_c'] == -999.0), 'water_temp_c'] = np.NaN

#Check minimum values to confirm change
print(stream['water_temp_c'].min())
# Calculate mean
stream['water_temp_c'].mean()

10.2


15.226666666666668

## 3 d

In [19]:
stream.iloc[18]

IndexError: single positional indexer is out-of-bounds

In [20]:
stream.iloc[17]

site_id               SB-03
date             2024-06-15
water_temp_c            NaN
discharge_cfs           180
nitrate_mg_l           0.05
qc_flag             suspect
Name: 17, dtype: object

In [23]:
pd.nan

AttributeError: module 'pandas' has no attribute 'nan'